# 01. I-JEPA multi-block masking

목표: 논문의 기본 비율을 작은 patch grid에 적용하고, target을 먼저 선택한 뒤 context에서 겹침을 제거하는 순서를 구현한다. 외부 패키지는 필요 없다. 이 코드는 개념 실습이며 공식 구현과 동일한 sampler가 아니다.

In [ ]:
import math
import random

def block_cells(top, left, height, width):
    return {
        (row, col)
        for row in range(top, top + height)
        for col in range(left, left + width)
    }

def sample_block(rng, grid_size, scale, aspect_ratio):
    # 면적 비율과 종횡비에서 정수 grid 크기를 구한다.
    area = max(1, round(grid_size * grid_size * scale))
    height = min(grid_size, max(1, round(math.sqrt(area * aspect_ratio))))
    width = min(grid_size, max(1, round(math.sqrt(area / aspect_ratio))))
    top = rng.randint(0, grid_size - height)
    left = rng.randint(0, grid_size - width)
    return block_cells(top, left, height, width)

In [ ]:
rng = random.Random(7)
grid_size = 16  # 224px 이미지와 14px patch를 단순화한 16×16 grid

targets = []
for _ in range(4):
    scale = rng.uniform(0.15, 0.20)
    aspect_ratio = rng.uniform(0.75, 1.50)
    targets.append(sample_block(rng, grid_size, scale, aspect_ratio))

raw_context = sample_block(
    rng,
    grid_size,
    scale=rng.uniform(0.85, 1.00),
    aspect_ratio=1.0,
)
all_target_cells = set().union(*targets)
context = raw_context - all_target_cells

print("전체 patch:", grid_size * grid_size)
print("target별 patch 수:", [len(target) for target in targets])
print("초기 context patch:", len(raw_context))
print("겹침 제거 후 context patch:", len(context))
assert context.isdisjoint(all_target_cells)

In [ ]:
# C=context, 숫자=target 번호, .=사용하지 않는 patch
for row in range(grid_size):
    symbols = []
    for col in range(grid_size):
        cell = (row, col)
        symbol = "C" if cell in context else "."
        for index, target in enumerate(targets, start=1):
            if cell in target:
                symbol = str(index)
        symbols.append(symbol)
    print(" ".join(symbols))

## 관찰 과제

1. seed를 바꾸고 실제 context 비율의 분포를 기록한다.
2. target scale을 0.02~0.05로 줄였을 때 semantic target이라는 직관이 유지되는지 논의한다.
3. target끼리는 겹칠 수 있지만 context와 target은 겹치지 않는 이유를 설명한다.
4. 공식 `src/masks/multiblock.py`와 비교해 batch 내 mask 길이를 맞추는 로직을 찾아본다.